# Goal 

- Write a prompt that will assist users in writing Python code, JSON config, or Regular Expressions focused on AWS-specific use cases  

- Input User will request code for a specific task 

- Output Python, JSON, or a regular expression without any explanation



```python
prompt = f"""
please provide a solution of the following task:

{task}

"""

```

# Load Env Variables and Create Client

In [32]:


from dotenv import load_dotenv

load_dotenv()


#Anthropic SDK
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [33]:
def add_user_message(messages, text):
    user_message = {"role":"user", "content" : text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content" : text}
    messages.append(assistant_message)
    
def chat(messages,system=None, temperature = 1.0, stop_sequences=[]):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature" : temperature,
        "stop_sequences" : stop_sequences
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

# Prompt Evaluation - generating dataset


In [34]:
import json
def generate_dataset():
  prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
      },
      ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
  """

  messages = []
  add_user_message(messages, prompt) 
  add_assistant_message(messages,"```json") 
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [35]:
dataset = generate_dataset()

with open("dataset.json" , "w") as f:
    json.dump(dataset, f, indent = 2)

# Prompt Evaluation - running the test dataset

In [36]:
def run_prompt(test_case):
    """Merges the prompt and test case input , then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """

    messages = []

    add_user_message(messages, prompt)
    output = chat(messages)

    return output

In [37]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #TODO - GRADING

    score = 10

    return {
        "output" : output,
        "test_case" : test_case,
        "score" : score
    }

In [38]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

[{'output': '# AWS S3 URI Parser\n\nHere\'s a Python function that parses S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri: str) -> dict:\n    """\n    Parses an AWS S3 bucket URI and returns bucket name and key.\n    \n    Args:\n        uri (str): S3 URI in the format \'s3://bucket-name/path/to/file\'\n        \n    Returns:\n        dict: Dictionary with \'bucket\' and \'key\' keys\n        \n    Raises:\n        ValueError: If the URI is invalid\n        \n    Example:\n        >>> parse_s3_uri(\'s3://my-bucket/path/to/file.txt\')\n        {\'bucket\': \'my-bucket\', \'key\': \'path/to/file.txt\'}\n    """\n    if not isinstance(uri, str):\n        raise ValueError("URI must be a string")\n    \n    if not uri.startswith(\'s3://\'):\n        raise ValueError("URI must start with \'s3://\'")\n    \n    # Remove the \'s3://\' prefix\n    uri_without_scheme = uri[5:]\n    \n    # Split by the first \'/\' to separate bucket from key\n    parts = uri_without_scheme.split(\'/\', 1)\n  

In [40]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a Python function that parses S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri: str) -> dict:\n    \"\"\"\n    Parses an AWS S3 bucket URI and returns bucket name and key.\n    \n    Args:\n        uri (str): S3 URI in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' keys\n        \n    Raises:\n        ValueError: If the URI is invalid\n        \n    Example:\n        >>> parse_s3_uri('s3://my-bucket/path/to/file.txt')\n        {'bucket': 'my-bucket', 'key': 'path/to/file.txt'}\n    \"\"\"\n    if not isinstance(uri, str):\n        raise ValueError(\"URI must be a string\")\n    \n    if not uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_scheme = uri[5:]\n    \n    # Split by the first '/' to separate bucket from key\n    parts = uri_without_scheme.split('/', 1)\n    \n    b